# 🔌 Notebook 07: Introduction to MCP (Model Context Protocol)

**Time:** 30 minutes  
**Goal:** Learn to connect LLMs to external tools and data using MCP

## What is MCP?

MCP (Model Context Protocol) is an **open protocol** that enables seamless integration between LLM applications and external data sources and tools.

Think of it as **USB for AI** - a standardized way to connect your LLM to:
- 📁 File systems
- 🔍 Search engines
- 💾 Databases
- 🌐 APIs
- 🛠️ Custom tools

## Why MCP Matters

**Without MCP:**
```
LLM → Hard-coded API calls → Brittle, custom code for each tool
```

**With MCP:**
```
LLM → MCP Protocol → Any MCP Server (modular, reusable)
```

## Key Concepts

### 1. MCP Server
A program that exposes tools/data to LLMs via the MCP protocol.

**Examples:**
- `brave-search` - Web search capabilities
- `filesystem` - File reading/writing
- `github` - Repository operations
- Custom servers you build!

### 2. MCP Client
Your application that connects to MCP servers (that's what we'll build!)

### 3. Tools
Functions that the LLM can call through MCP servers.

**Example tool:**
```json
{
  "name": "search_web",
  "description": "Search the web for information",
  "parameters": {
    "query": "string"
  }
}
```

## What You'll Learn

- Understanding MCP architecture
- Setting up MCP servers
- Calling tools from Python
- Building agent workflows
- Creating your own MCP tools
- Best practices for tool use

Let's build agents that can DO things! 🚀

**Prerequisites:** Notebooks 02-06 completed, Node.js installed (for MCP servers)

In [1]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json
import subprocess
from typing import Dict, List, Any, Optional

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, append_to_reflection
from src.config import PATH

print("=" * 60)
print("NOTEBOOK 07: INTRODUCTION TO MCP")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

# Check Node.js installation (required for MCP servers)
print("Checking prerequisites...")
print("-" * 60)

try:
    node_version = subprocess.check_output(['node', '--version'], 
                                          stderr=subprocess.STDOUT, 
                                          text=True).strip()
    print(f"✓ Node.js installed: {node_version}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("✗ Node.js NOT found!")
    print("  MCP servers require Node.js")
    print("  Install from: https://nodejs.org/")
    print()
    print("  ⚠️  Some examples in this notebook may not work without Node.js")

try:
    npx_check = subprocess.check_output(['npx', '--version'], 
                                       stderr=subprocess.STDOUT, 
                                       text=True).strip()
    print(f"✓ npx available: {npx_check}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("✗ npx NOT found (should come with Node.js)")

print()
print("✓ Ready to learn MCP!")
print()

NOTEBOOK 07: INTRODUCTION TO MCP

Configuration loaded: Path C

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5
✓ Ollama client initialized
  Available models: ['llama3.2:latest', 'llama2:latest', 'llama3:latest']
  Default model: llama3.2:latest
Checking prerequisites...
------------------------------------------------------------
✓ Node.js installed: v22.21.0
✗ npx NOT found (should come with Node.js)

✓ Ready to learn MCP!



---
## 📐 Part 1: MCP Architecture

Let's understand how MCP works before we use it.

In [2]:
# MCP Architecture Explanation
print("=" * 60)
print("MCP ARCHITECTURE")
print("=" * 60)
print()

architecture = """
┌─────────────────────────────────────────────────────────────┐
│                        YOUR APPLICATION                      │
│  ┌─────────────┐         ┌──────────────┐                  │
│  │     LLM     │ ◄─────► │  MCP Client  │                  │
│  │  (Claude)   │         │  (Your Code) │                  │
│  └─────────────┘         └──────────────┘                  │
│                                 │                            │
└─────────────────────────────────┼────────────────────────────┘
                                  │ MCP Protocol
                                  │ (JSON-RPC over stdio)
                ┌─────────────────┼─────────────────┐
                │                 │                 │
        ┌───────▼──────┐  ┌───────▼──────┐  ┌─────▼──────┐
        │ MCP Server 1 │  │ MCP Server 2 │  │ MCP Server │
        │ (brave-search)│  │ (filesystem) │  │  (custom)  │
        └──────────────┘  └──────────────┘  └────────────┘
               │                 │                  │
        ┌──────▼──────┐   ┌──────▼──────┐   ┌──────▼──────┐
        │ Web Search  │   │ Read/Write  │   │ Your Tools  │
        │    API      │   │   Files     │   │             │
        └─────────────┘   └─────────────┘   └─────────────┘

KEY CONCEPTS:

1. MCP Client: Your Python code that manages connections to servers
2. MCP Server: A program that exposes tools via MCP protocol
3. Tool: A function the LLM can call (like "search_web", "read_file")
4. Protocol: Standardized JSON-RPC messages over stdin/stdout

WORKFLOW:
1. LLM decides it needs to use a tool
2. LLM outputs a tool call in JSON format
3. MCP Client sends request to appropriate MCP Server
4. MCP Server executes the tool and returns result
5. Result goes back to LLM
6. LLM continues reasoning with the new information
"""

print(architecture)
print()

print("💡 Key Insight: MCP makes LLMs **agentic** - they can take actions!")
print()

MCP ARCHITECTURE


┌─────────────────────────────────────────────────────────────┐
│                        YOUR APPLICATION                      │
│  ┌─────────────┐         ┌──────────────┐                  │
│  │     LLM     │ ◄─────► │  MCP Client  │                  │
│  │  (Claude)   │         │  (Your Code) │                  │
│  └─────────────┘         └──────────────┘                  │
│                                 │                            │
└─────────────────────────────────┼────────────────────────────┘
                                  │ MCP Protocol
                                  │ (JSON-RPC over stdio)
                ┌─────────────────┼─────────────────┐
                │                 │                 │
        ┌───────▼──────┐  ┌───────▼──────┐  ┌─────▼──────┐
        │ MCP Server 1 │  │ MCP Server 2 │  │ MCP Server │
        │ (brave-search)│  │ (filesystem) │  │  (custom)  │
        └──────────────┘  └──────────────┘  └────────────┘
               │  

---
## 🛠️ Part 2: MCP Server Configuration

MCP servers are configured in Claude Desktop's config file. Let's see how.

In [3]:
# Understanding MCP Configuration
print("=" * 60)
print("MCP SERVER CONFIGURATION")
print("=" * 60)
print()

config_example = """
MCP servers are configured in:
  macOS: ~/Library/Application Support/Claude/claude_desktop_config.json
  Windows: %APPDATA%\\Claude\\claude_desktop_config.json

Example configuration:
{
  "mcpServers": {
    "brave-search": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-brave-search"
      ],
      "env": {
        "BRAVE_API_KEY": "BRAVE_API_KEY"
      }
    },
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/username/Desktop"
      ]
    },
    "sequential-thinking": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-sequential-thinking"
      ]
    }
  }
}

CONFIGURATION FIELDS:
- "command": The program to run (usually "npx" for Node-based servers)
- "args": Command-line arguments, including the server package name
- "env": Environment variables (like API keys)

IMPORTANT: After editing config, restart Claude Desktop!
"""

print(config_example)
print()

# Try to detect Claude Desktop config
home = Path.home()
if sys.platform == "darwin":  # macOS
    config_path = home / "Library/Application Support/Claude/claude_desktop_config.json"
elif sys.platform == "win32":  # Windows
    config_path = Path(os.getenv('APPDATA')) / "Claude/claude_desktop_config.json"
else:  # Linux
    config_path = home / ".config/Claude/claude_desktop_config.json"

print("Checking for Claude Desktop config...")
print("-" * 60)
if config_path.exists():
    print(f"✓ Found config at: {config_path}")
    print()
    print("To add MCP servers:")
    print(f"  1. Edit: {config_path}")
    print("  2. Add server configuration (see example above)")
    print("  3. Restart Claude Desktop")
else:
    print(f"✗ Config not found at: {config_path}")
    print()
    print("This is OK if:")
    print("  • You haven't installed Claude Desktop")
    print("  • You're running this in a different environment")
    print()
    print("To use MCP in Claude Desktop:")
    print("  1. Install Claude Desktop from claude.ai")
    print("  2. Create the config file at the path above")
    print("  3. Add your MCP servers to the config")

print()

MCP SERVER CONFIGURATION


MCP servers are configured in:
  macOS: ~/Library/Application Support/Claude/claude_desktop_config.json
  Windows: %APPDATA%\Claude\claude_desktop_config.json

Example configuration:
{
  "mcpServers": {
    "brave-search": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-brave-search"
      ],
      "env": {
        "BRAVE_API_KEY": "BRAVE_API_KEY"
      }
    },
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/username/Desktop"
      ]
    },
    "sequential-thinking": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-sequential-thinking"
      ]
    }
  }
}

CONFIGURATION FIELDS:
- "command": The program to run (usually "npx" for Node-based servers)
- "args": Command-line arguments, including the server package name
- "env": Environment variables (like API keys)

IMPORTANT: After

---
## 🔍 Part 3: Available MCP Servers

Let's explore some official MCP servers you can use.

In [4]:
# MCP Server Catalog
print("=" * 60)
print("POPULAR MCP SERVERS")
print("=" * 60)
print()

mcp_servers = {
    "Search & Information": {
        "brave-search": {
            "package": "@modelcontextprotocol/server-brave-search",
            "description": "Web search using Brave Search API",
            "requires": "BRAVE_API_KEY",
            "use_case": "Finding current information, research"
        },
        "fetch": {
            "package": "@modelcontextprotocol/server-fetch",
            "description": "Fetch and extract content from URLs",
            "requires": "None",
            "use_case": "Reading web pages, accessing APIs"
        }
    },
    
    "File & Data": {
        "filesystem": {
            "package": "@modelcontextprotocol/server-filesystem",
            "description": "Read/write files in specified directories",
            "requires": "Directory path as argument",
            "use_case": "File management, data processing"
        },
        "sqlite": {
            "package": "@modelcontextprotocol/server-sqlite",
            "description": "Query SQLite databases",
            "requires": "Database path as argument",
            "use_case": "Data analysis, database queries"
        }
    },
    
    "Development": {
        "github": {
            "package": "@modelcontextprotocol/server-github",
            "description": "Interact with GitHub repositories",
            "requires": "GITHUB_PERSONAL_ACCESS_TOKEN",
            "use_case": "Code review, repo management"
        },
        "git": {
            "package": "@modelcontextprotocol/server-git",
            "description": "Git operations on local repositories",
            "requires": "Repository path as argument",
            "use_case": "Version control, commit history"
        }
    },
    
    "Utilities": {
        "sequential-thinking": {
            "package": "@modelcontextprotocol/server-sequential-thinking",
            "description": "Enhanced reasoning through structured thinking",
            "requires": "None",
            "use_case": "Complex problem solving"
        },
        "memory": {
            "package": "@modelcontextprotocol/server-memory",
            "description": "Persistent memory across conversations",
            "requires": "None",
            "use_case": "Maintaining context, user preferences"
        }
    }
}

for category, servers in mcp_servers.items():
    print(f"📁 {category}")
    print("=" * 60)
    
    for name, info in servers.items():
        print(f"\n🔧 {name}")
        print(f"   Package: {info['package']}")
        print(f"   Description: {info['description']}")
        print(f"   Requires: {info['requires']}")
        print(f"   Use Case: {info['use_case']}")
    
    print()

print()
print("💡 Install any server with: npx -y <package-name>")
print("   Full list: https://github.com/modelcontextprotocol/servers")
print()

POPULAR MCP SERVERS

📁 Search & Information

🔧 brave-search
   Package: @modelcontextprotocol/server-brave-search
   Description: Web search using Brave Search API
   Requires: BRAVE_API_KEY
   Use Case: Finding current information, research

🔧 fetch
   Package: @modelcontextprotocol/server-fetch
   Description: Fetch and extract content from URLs
   Requires: None
   Use Case: Reading web pages, accessing APIs

📁 File & Data

🔧 filesystem
   Package: @modelcontextprotocol/server-filesystem
   Description: Read/write files in specified directories
   Requires: Directory path as argument
   Use Case: File management, data processing

🔧 sqlite
   Package: @modelcontextprotocol/server-sqlite
   Description: Query SQLite databases
   Requires: Database path as argument
   Use Case: Data analysis, database queries

📁 Development

🔧 github
   Package: @modelcontextprotocol/server-github
   Description: Interact with GitHub repositories
   Requires: GITHUB_PERSONAL_ACCESS_TOKEN
   Use Case: C

---
## 🧪 Part 4: Tool Use in Practice

Now let's see how LLMs actually use tools through the MCP protocol.

### Tool Calling Flow

When an LLM has access to tools, here's what happens:

**1. User Request:**
```
"Search for recent papers on transformer architectures"
```

**2. LLM Recognizes Need for Tool:**
```json
{
  "type": "tool_use",
  "name": "brave_web_search",
  "input": {
    "query": "transformer architecture papers 2024"
  }
}
```

**3. Tool Executes & Returns Result:**
```json
{
  "type": "tool_result",
  "content": "Found 10 papers: 1. 'Attention is All You Need' ..."
}
```

**4. LLM Continues with Result:**
```
"Based on the search results, here are the most relevant papers..."
```

In [5]:
# Simulating Tool Use
print("=" * 60)
print("EXPERIMENT 1: Understanding Tool Use")
print("=" * 60)
print()

# Simulate what happens when LLM has access to tools
# (In real MCP, tools are automatically available to Claude Desktop)

system_prompt_with_tools = """You are an AI assistant with access to tools.

Available tools:
- search_web(query: str) -> str: Search the web for information
- read_file(path: str) -> str: Read contents of a file
- calculate(expression: str) -> float: Perform calculations

When you need to use a tool, output JSON in this format:
{
  "tool": "tool_name",
  "parameters": {
    "param1": "value1"
  }
}

After using a tool, you'll receive the result and can continue reasoning."""

user_request = """I need to find the latest information about GPT-4's capabilities 
and then calculate how many parameters it has relative to GPT-3."""

print("System Prompt with Tools:")
print("-" * 60)
print(system_prompt_with_tools)
print()

print("User Request:")
print("-" * 60)
print(user_request)
print()

response = client.generate(
    prompt=user_request,
    system=system_prompt_with_tools,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response:
    print("LLM Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
    
    print()
    print("💡 Notice how the LLM:")
    print("   1. Recognizes it needs external information")
    print("   2. Chooses the right tool (search_web)")
    print("   3. Formats the tool call correctly")
    print("   4. Would continue reasoning after getting results")
else:
    print(f"Error: {response['error']}")

print()

EXPERIMENT 1: Understanding Tool Use

System Prompt with Tools:
------------------------------------------------------------
You are an AI assistant with access to tools.

Available tools:
- search_web(query: str) -> str: Search the web for information
- read_file(path: str) -> str: Read contents of a file
- calculate(expression: str) -> float: Perform calculations

When you need to use a tool, output JSON in this format:
{
  "tool": "tool_name",
  "parameters": {
    "param1": "value1"
  }
}

After using a tool, you'll receive the result and can continue reasoning.

User Request:
------------------------------------------------------------
I need to find the latest information about GPT-4's capabilities 
and then calculate how many parameters it has relative to GPT-3.

LLM Response:
To get the latest information about GPT-4's capabilities, I will search the web for this query:

{
  "tool": "search_web",
  "parameters": {
    "query": "latest news gpt-4 capabilities"
  }
}

Please wait

---
## 🔨 Part 5: Building Your First MCP Tool

Let's create a simple custom MCP server with a useful tool.

In [6]:
# Custom MCP Server Example
print("=" * 60)
print("BUILDING A CUSTOM MCP SERVER")
print("=" * 60)
print()

custom_server_code = '''
Example: Simple Research Helper MCP Server

This would be saved as a Node.js file and run as an MCP server.
For this notebook, we'll show the concept (actual implementation 
would be in a separate .js file).

// research-helper-mcp.js
import { Server } from "@modelcontextprotocol/sdk/server/index.js";
import { StdioServerTransport } from "@modelcontextprotocol/sdk/server/stdio.js";

const server = new Server({
  name: "research-helper",
  version: "1.0.0"
});

// Define a tool
server.setRequestHandler("tools/list", async () => {
  return {
    tools: [
      {
        name: "extract_citations",
        description: "Extract citations from a research paper text",
        inputSchema: {
          type: "object",
          properties: {
            text: {
              type: "string",
              description: "The paper text to extract citations from"
            }
          },
          required: ["text"]
        }
      },
      {
        name: "generate_summary",
        description: "Generate a structured summary of a paper",
        inputSchema: {
          type: "object",
          properties: {
            title: { type: "string" },
            abstract: { type: "string" }
          },
          required: ["title", "abstract"]
        }
      }
    ]
  };
});

// Handle tool calls
server.setRequestHandler("tools/call", async (request) => {
  const { name, arguments: args } = request.params;
  
  if (name === "extract_citations") {
    // Simple citation extraction logic
    const citations = args.text.match(/\\[\\d+\\]/g) || [];
    return {
      content: [{
        type: "text",
        text: `Found ${citations.length} citations: ${citations.join(", ")}`
      }]
    };
  }
  
  if (name === "generate_summary") {
    return {
      content: [{
        type: "text",
        text: `Summary for "${args.title}":\\n\\nAbstract: ${args.abstract}`
      }]
    };
  }
});

// Start server
const transport = new StdioServerTransport();
await server.connect(transport);

TO USE THIS SERVER:
1. Save the code above to: research-helper-mcp.js
2. Add to claude_desktop_config.json:
   {
     "mcpServers": {
       "research-helper": {
         "command": "node",
         "args": ["path/to/research-helper-mcp.js"]
       }
     }
   }
3. Restart Claude Desktop
4. The tools will be automatically available!
'''

print(custom_server_code)
print()

print("=" * 60)
print("KEY COMPONENTS OF AN MCP SERVER")
print("=" * 60)
print()

components = {
    "1. Server Setup": "Initialize the MCP server with name and version",
    "2. Tool Definitions": "List available tools with descriptions and schemas",
    "3. Tool Handlers": "Implement the actual logic for each tool",
    "4. Transport": "Communication channel (usually stdio for local servers)",
    "5. Connection": "Start the server and listen for requests"
}

for component, description in components.items():
    print(f"{component}")
    print(f"   {description}")
    print()

print("💡 The MCP SDK handles all the protocol details - you just define tools!")
print()

BUILDING A CUSTOM MCP SERVER


Example: Simple Research Helper MCP Server

This would be saved as a Node.js file and run as an MCP server.
For this notebook, we'll show the concept (actual implementation 
would be in a separate .js file).

// research-helper-mcp.js
import { Server } from "@modelcontextprotocol/sdk/server/index.js";
import { StdioServerTransport } from "@modelcontextprotocol/sdk/server/stdio.js";

const server = new Server({
  name: "research-helper",
  version: "1.0.0"
});

// Define a tool
server.setRequestHandler("tools/list", async () => {
  return {
    tools: [
      {
        name: "extract_citations",
        description: "Extract citations from a research paper text",
        inputSchema: {
          type: "object",
          properties: {
            text: {
              type: "string",
              description: "The paper text to extract citations from"
            }
          },
          required: ["text"]
        }
      },
      {
        name: "generat

---
## 🎯 Your Turn: Practice Tasks

Time to work with MCP concepts!

### 📝 Task 1: Design Your Research Agent Tools

**Goal:** Design the MCP tools your research agent will need.

In [7]:
# TODO - Task 1: Design Your Tools
print("=" * 60)
print("TASK 1: Design Your Research Agent Tools")
print("=" * 60)
print()

# ============================================================================
# TODO: Design 3-5 tools your research agent needs
# ============================================================================

your_tools = """
Tool 1: search_academic_sources
  Description: Search academic databases (e.g., Google Scholar, arXiv, PubMed) for relevant research papers
  Parameters:
    - query (string): Research keywords or question
    - max_results (integer): Number of results to return (default: 10)
    - year_filter (integer, optional): Only return papers after this year
  Returns: List of papers with title, authors, abstract, publication year, and URL
  Use Case: Initial discovery of credible academic sources

Tool 2: summarize_document
  Description: Generate a concise summary of a long document or research paper
  Parameters:
    - text (string): Full document or extracted content
    - max_length (integer, optional): Desired summary length
  Returns: Structured summary with key points and main conclusions
  Use Case: Quickly understanding long papers or reports

Tool 3: extract_key_insights
  Description: Identify key findings, methodologies, and conclusions from research content
  Parameters:
    - text (string): Input research content
  Returns: 
    - key_findings (list)
    - methodology (string)
    - limitations (list)
  Use Case: Deep analysis phase to extract meaningful insights

Tool 4: verify_source_credibility
  Description: Evaluate the credibility and reliability of a source
  Parameters:
    - source_url (string): URL of the source
    - metadata (optional dict): Author, publisher, date
  Returns:
    - credibility_score (0-100)
    - reasoning (string)
  Use Case: Filtering trustworthy vs low-quality sources

Tool 5: generate_citations
  Description: Generate properly formatted citations (APA, MLA, Chicago)
  Parameters:
    - source_data (dict): Includes title, authors, year, publisher, URL
    - style (string): Citation style (APA, MLA, Chicago)
  Returns: Formatted citation string
  Use Case: Final report writing and referencing
"""

print(your_tools)
print()

# ============================================================================
# TODO: Map tools to your workflow
# ============================================================================

workflow_mapping = """
Workflow Step 1: Define research question and discover sources
  Tools needed: [search_academic_sources]
  Why: This step focuses on gathering relevant and credible research materials to build the foundation of the analysis.

Workflow Step 2: Filter and validate source quality
  Tools needed: [verify_source_credibility]
  Why: Ensures that only high-quality, trustworthy sources are used before investing time in deeper analysis.

Workflow Step 3: Read and condense content
  Tools needed: [summarize_document]
  Why: Long research papers are condensed into digestible summaries to speed up understanding.

Workflow Step 4: Extract insights and analyze findings
  Tools needed: [extract_key_insights]
  Why: Identifies key findings, methodologies, and limitations to support critical thinking and synthesis.

Workflow Step 5: Compile results and format references
  Tools needed: [generate_citations]
  Why: Final output requires properly formatted citations and organized insights for reporting.
"""

print("=" * 60)
print("WORKFLOW MAPPING")
print("=" * 60)
print()
print(workflow_mapping)

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Tool Design

**Total tools designed:** 5

**Categories:**
- Search/Discovery: 1 tool
- Data Processing: 1 tool
- Analysis: 2 tools
- Output Generation: 1 tool

### Tool Complexity

**Simplest tool:** generate_citations  
This tool is simple because it mainly formats structured input into predefined citation styles without requiring complex reasoning or external data fetching.

**Most complex tool:** verify_source_credibility  
This tool is complex because it requires evaluating multiple factors (author reputation, publication source, recency, bias), potentially involving external APIs and heuristic or ML-based scoring.

### Existing vs Custom

**Tools that exist (can use off-the-shelf):** 
- search_academic_sources (via APIs like Google Scholar, arXiv, PubMed)
- summarize_document (LLM-based APIs)
- generate_citations (libraries like citeproc, bibtex tools)

**Tools you'd need to build custom:**
- verify_source_credibility
- extract_key_insights

### Implementation Priority

**Must-have (Priority 1):**
1. search_academic_sources
2. summarize_document

**Nice-to-have (Priority 2):**
1. extract_key_insights
2. generate_citations

**Future enhancement (Priority 3):**
1. verify_source_credibility

### Dependencies

**Do your tools depend on each other?**
Yes — there is a clear dependency chain. For example, summarize_document and extract_key_insights both depend on outputs from search_academic_sources.

**What's the sequence of tool calls?**
1. search_academic_sources → retrieve papers  
2. verify_source_credibility → filter results  
3. summarize_document → condense content  
4. extract_key_insights → analyze findings  
5. generate_citations → format references  

### Data Flow

**What data flows between tools?**
- search_academic_sources outputs paper metadata and abstracts  
- These are passed to verify_source_credibility for filtering  
- Selected full texts are passed to summarize_document  
- Summaries or full text feed into extract_key_insights  
- Metadata is passed to generate_citations  

**Any bottlenecks?**
- Access to full-text papers (paywalls)
- Latency in summarizing long documents
- Incomplete metadata affecting citation quality

### Error Handling

**What could go wrong with each tool?**
- search_academic_sources: no results or irrelevant results  
- summarize_document: loss of important context  
- extract_key_insights: missing or misinterpreting key findings  
- verify_source_credibility: inaccurate scoring  
- generate_citations: incorrect formatting due to missing fields  

**How would you handle failures?**
- Add fallback queries or broaden search scope  
- Allow user review or multiple summary versions  
- Cross-check insights with multiple sources  
- Use hybrid scoring (rules + ML) for credibility  
- Validate citation inputs and prompt for missing data  

### Real-World Readiness

**Could you actually build these tools?** Yes (with some limitations)

**What skills/resources would you need?**
- API integration (REST APIs for search engines)
- NLP/LLM usage for summarization and extraction
- Basic ML or heuristics for credibility scoring
- Knowledge of citation formats and libraries
- Handling rate limits and data cleaning

**Timeline to implement:**
- search_academic_sources: 1–2 days  
- summarize_document: 1 day  
- extract_key_insights: 2–3 days  
- verify_source_credibility: 3–5 days  
- generate_citations: 0.5–1 day  

### Biggest Insight

Designing tools for agents requires balancing modularity and dependency. Simple, reusable tools are powerful when chained together, but the overall system becomes fragile if one step fails—so robustness and fallback strategies are just as important as functionality.
"""

print(reflection)

append_to_reflection(
    notebook="07",
    section_title="Task 1 - Design Your Research Agent Tools",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 1: Design Your Research Agent Tools


Tool 1: search_academic_sources
  Description: Search academic databases (e.g., Google Scholar, arXiv, PubMed) for relevant research papers
  Parameters:
    - query (string): Research keywords or question
    - max_results (integer): Number of results to return (default: 10)
    - year_filter (integer, optional): Only return papers after this year
  Returns: List of papers with title, authors, abstract, publication year, and URL
  Use Case: Initial discovery of credible academic sources

Tool 2: summarize_document
  Description: Generate a concise summary of a long document or research paper
  Parameters:
    - text (string): Full document or extracted content
    - max_length (integer, optional): Desired summary length
  Returns: Structured summary with key points and main conclusions
  Use Case: Quickly understanding long papers or reports

Tool 3: extract_key_insights
  Description: Identify key findings, methodologies, and conclusions fro

### 📝 Task 2: Tool Calling Strategy

**Goal:** Understand when and how your agent should call tools.

In [8]:
# TODO - Task 2: Tool Calling Strategy
print("=" * 60)
print("TASK 2: Tool Calling Strategy")
print("=" * 60)
print()

# Scenario: Design how your agent decides which tools to use
scenario = """
SCENARIO: Your research agent receives this request:

"Find recent papers on transformer architectures, read the top 3, 
extract their key contributions, and create a comparison table."

This requires multiple tool calls in sequence. How should your agent:
1. Decide which tools to use?
2. Determine the order?
3. Know when to stop?
4. Handle errors?
"""

print(scenario)
print()

# ============================================================================
# TODO: Design your tool calling strategy
# ============================================================================

your_strategy = """
# YOUR TOOL CALLING STRATEGY

## Decision Tree

IF user request contains research-related keywords (e.g., "papers", "studies", "research", "evidence"):
  → Use tool: search_academic_sources
  → Because: Need to gather relevant academic sources as a foundation

IF user provides a long document or paper:
  → Use tool: summarize_document
  → Because: Condense content for faster understanding

IF summarized or raw research content is available:
  → Use tool: extract_key_insights
  → Because: Identify key findings, methodology, and limitations

IF sources are retrieved from search:
  → Use tool: verify_source_credibility
  → Because: Filter out low-quality or unreliable sources before analysis

IF user asks for references or final report:
  → Use tool: generate_citations
  → Because: Format sources into proper citation styles

IF previous tool returned filtered/validated sources:
  → Next tool: summarize_document
  → Because: Move from discovery to understanding

IF previous tool returned summaries:
  → Next tool: extract_key_insights
  → Because: Move from understanding to analysis

IF previous tool returned structured insights:
  → Next tool: generate_citations
  → Because: Prepare final output for reporting

## Example Flow for Scenario

Step 1: User asks for research on "impact of AI in healthcare"
  Tool: search_academic_sources
  Input: query="impact of AI in healthcare", max_results=10
  Output: List of research papers with metadata

Step 2: Filter results
  Tool: verify_source_credibility
  Input: source_url + metadata from Step 1
  Output: credibility scores and filtered list

Step 3: Summarize selected papers
  Tool: summarize_document
  Input: full text or abstracts from Step 2
  Output: concise summaries

Step 4: Extract insights
  Tool: extract_key_insights
  Input: summaries or full text
  Output: key findings, methodology, limitations

Step 5: Generate references
  Tool: generate_citations
  Input: metadata from Step 1/2
  Output: formatted citations (APA/MLA/etc.)

## Error Handling

IF tool call fails:
  → Action: Retry once, then fallback to alternative or skip
  → Reasoning: Temporary issues (API, timeout) may resolve, otherwise maintain workflow continuity

IF tool returns empty results:
  → Action: Broaden query or modify keywords
  → Reasoning: Avoid dead-ends and improve recall

IF summarize_document fails or is low quality:
  → Action: Reduce input size or chunk document
  → Reasoning: Large inputs may degrade performance

IF verify_source_credibility is uncertain:
  → Action: Use heuristic fallback (e.g., prioritize well-known journals)
  → Reasoning: Maintain minimum quality filtering

## Stopping Conditions

The agent should stop calling tools when:
1. Sufficient high-quality information has been gathered and analyzed
2. The user’s question has been fully answered with supporting evidence
3. Additional tool calls provide diminishing or redundant value
"""

print("=" * 60)
print("YOUR STRATEGY")
print("=" * 60)
print(your_strategy)

# ============================================================================
# TODO: Simulate the tool calling sequence
# ============================================================================

print()
print("=" * 60)
print("SIMULATED EXECUTION")
print("=" * 60)
print()

simulation = """
# SIMULATE HOW YOUR AGENT WOULD HANDLE THE SCENARIO

Request: "Find recent papers on transformer architectures, 
         read the top 3, extract their key contributions, 
         and create a comparison table."

Call 1:
  Tool: search_academic_sources
  Input: {query: "transformer architectures recent papers", max_results: 5, year_filter: 2022}
  Output: [
    {title: "Efficient Transformers: A Survey", authors: "...", year: 2023, url: "..."},
    {title: "FlashAttention: Fast and Memory-Efficient Attention", authors: "...", year: 2022, url: "..."},
    {title: "Scaling Vision Transformers", authors: "...", year: 2023, url: "..."}
  ]
  Agent's reasoning: Start by retrieving the most relevant and recent papers. Next, ensure they are credible.

Call 2:
  Tool: verify_source_credibility
  Input: {source_url + metadata from Call 1}
  Output: [
    {title: "...Efficient Transformers...", score: 92},
    {title: "...FlashAttention...", score: 95},
    {title: "...Scaling Vision Transformers...", score: 90}
  ]
  Agent's reasoning: All top 3 sources are credible. Proceed to understanding their content.

Call 3:
  Tool: summarize_document
  Input: {text: abstracts/full text of top 3 papers}
  Output: [
    {title: "Efficient Transformers...", summary: "Overview of optimization techniques..."},
    {title: "FlashAttention...", summary: "Introduces memory-efficient attention algorithm..."},
    {title: "Scaling Vision Transformers...", summary: "Explores scaling strategies for ViTs..."}
  ]
  Agent's reasoning: Summaries provide quick understanding. Next step is deeper analysis.

Call 4:
  Tool: extract_key_insights
  Input: {text: summaries from Call 3}
  Output: [
    {
      title: "Efficient Transformers...",
      key_findings: ["Taxonomy of efficient methods", "Benchmark comparisons"],
      methodology: "Literature survey",
      limitations: ["Lacks experimental validation"]
    },
    {
      title: "FlashAttention...",
      key_findings: ["Reduces memory usage", "Improves speed significantly"],
      methodology: "Algorithm design + experiments",
      limitations: ["Hardware-specific optimizations"]
    },
    {
      title: "Scaling Vision Transformers...",
      key_findings: ["Scaling laws for ViTs", "Performance improvements with size"],
      methodology: "Empirical scaling experiments",
      limitations: ["High compute cost"]
    }
  ]
  Agent's reasoning: Now we have structured insights. Next, format into a comparison table and add citations.

Call 5:
  Tool: generate_citations
  Input: {source_data: metadata from Call 1, style: "APA"}
  Output: [
    "Author (2023). Efficient Transformers: A Survey...",
    "Author (2022). FlashAttention: Fast and Memory-Efficient Attention...",
    "Author (2023). Scaling Vision Transformers..."
  ]
  Agent's reasoning: Citations prepared. Now combine everything into final output.

Final Output:

Comparison Table:

| Paper | Key Contributions | Methodology | Limitations |
|------|------------------|-------------|------------|
| Efficient Transformers: A Survey | Categorizes optimization techniques; benchmarks methods | Literature survey | Limited experimental validation |
| FlashAttention | Memory-efficient attention; faster computation | Algorithm + experiments | Hardware-dependent |
| Scaling Vision Transformers | Demonstrates scaling laws; improves performance | Empirical experiments | High compute requirements |

References (APA):
- Author (2023). Efficient Transformers: A Survey...
- Author (2022). FlashAttention...
- Author (2023). Scaling Vision Transformers...

Total Tool Calls: 5
Success: Yes
"""

print(simulation)

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Strategy Design

**How many tool calls did your scenario require?** 5

**Could it be done with fewer?** Yes  
If abstracts are sufficient, we could skip full summarization and directly extract insights from abstracts, reducing to ~3–4 calls.

**What's the maximum tool calls you'd allow?** 6–8  
This limit balances depth of analysis with latency and cost. Beyond this, diminishing returns and slower user experience become significant.

### Decision Logic

**How does your agent choose which tool to use?**
The agent uses a combination of keyword detection (intent classification) and workflow state (what data is already available). It decides based on:
- User intent (search, summarize, analyze, format)
- Output of previous tools (e.g., summaries trigger insight extraction)

**Is it rule-based or LLM-driven?**
Hybrid approach:
- Rule-based for workflow sequencing and control flow
- LLM-driven for interpreting user intent and handling ambiguous cases

**Advantages of your approach:**
1. Predictable and structured execution flow  
2. Flexible enough to handle varied user requests  

**Disadvantages:**
1. Rules may not generalize to all edge cases  
2. Requires careful tuning to avoid unnecessary tool usage  

### Error Resilience

**Most likely point of failure:** summarize_document  
(Since long or complex documents may lead to poor summaries)

**How would you make it robust?**
- Chunk large documents before summarization  
- Use multiple summaries and aggregate results  
- Allow fallback to abstract-only processing  

**Should the agent retry failed calls?** Yes  
Retry for transient errors (timeouts, API failures), but not for logical errors (e.g., empty input)

### Efficiency

**Any redundant tool calls in your flow?**
- verify_source_credibility may be redundant if sources are already from trusted repositories  
- summarize_document + extract_key_insights could sometimes be merged  

**How could you optimize?**
- Combine summarization and insight extraction into a single step  
- Skip credibility check for known high-quality sources  
- Limit number of papers dynamically based on relevance  

**Caching strategy:**
- Cache search results and summaries for 24 hours  
- Cache citations indefinitely (they rarely change)  
- Use query-based keys to reuse prior results  

### User Experience

**Should users see tool calls happening?** Partial  
Users don’t need technical details, but should see progress updates.

**How would you show progress?**
- “Searching for papers…”  
- “Analyzing top results…”  
- “Extracting insights and building comparison…”  

**What if tools are slow?**
- Show intermediate results (e.g., list of papers first)  
- Use streaming responses  
- Provide estimated wait time if possible  

### Production Concerns

**Rate limits:**
- Implement request throttling and batching  
- Cache frequent queries  
- Use fallback sources if primary API is limited  

**Costs:**
- Yes, tool calls add cost, but structured workflows reduce wasted computation  
- Optimization (fewer calls, caching) keeps it sustainable  

**Monitoring:**
- Tool latency and error rates  
- Success rate of workflows  
- User satisfaction (feedback/engagement)  
- Cost per request  

### Biggest Challenge

Balancing depth of analysis with efficiency—deciding when to stop calling tools without missing important insights.

### Key Insight

Agentic behavior is less about individual tool quality and more about orchestration—how well tools are sequenced, combined, and adapted dynamically determines the overall system effectiveness.
"""

print(reflection)

append_to_reflection(
    notebook="07",
    section_title="Task 2 - Tool Calling Strategy",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 2: Tool Calling Strategy


SCENARIO: Your research agent receives this request:

"Find recent papers on transformer architectures, read the top 3, 
extract their key contributions, and create a comparison table."

This requires multiple tool calls in sequence. How should your agent:
1. Decide which tools to use?
2. Determine the order?
3. Know when to stop?
4. Handle errors?


YOUR STRATEGY

# YOUR TOOL CALLING STRATEGY

## Decision Tree

IF user request contains research-related keywords (e.g., "papers", "studies", "research", "evidence"):
  → Use tool: search_academic_sources
  → Because: Need to gather relevant academic sources as a foundation

IF user provides a long document or paper:
  → Use tool: summarize_document
  → Because: Condense content for faster understanding

IF summarized or raw research content is available:
  → Use tool: extract_key_insights
  → Because: Identify key findings, methodology, and limitations

IF sources are retrieved from search:
  → Use tool: ve

### 📝 Task 3: MCP vs Alternatives

**Goal:** Understand when to use MCP vs other approaches.

In [9]:
# TODO - Task 3: MCP vs Alternatives
print("=" * 60)
print("TASK 3: MCP vs Alternatives")
print("=" * 60)
print()

# Compare different approaches to giving LLMs tools
approaches = {
    "MCP (Model Context Protocol)": {
        "how_it_works": "Standardized protocol, modular servers",
        "pros": [
            "Standardized - reusable across applications",
            "Modular - plug and play servers",
            "Growing ecosystem of servers",
            "Clean separation of concerns"
        ],
        "cons": [
            "Requires MCP infrastructure",
            "Still relatively new",
            "Need to learn MCP SDK"
        ],
        "best_for": "Production apps, reusable tools, Claude Desktop integration"
    },
    
    "Function Calling (API)": {
        "how_it_works": "Define functions in API call, LLM returns function calls",
        "pros": [
            "Built into Claude API",
            "No extra infrastructure",
            "Full control over execution",
            "Well documented"
        ],
        "cons": [
            "Custom code for each tool",
            "Not reusable across apps",
            "Have to handle all execution logic"
        ],
        "best_for": "API-based apps, custom tools, simple workflows"
    },
    
    "Prompt Engineering (Manual)": {
        "how_it_works": "Instruct LLM to output tool calls, parse manually",
        "pros": [
            "Works with any LLM",
            "No special setup",
            "Maximum flexibility"
        ],
        "cons": [
            "Unreliable parsing",
            "More prompt engineering",
            "Error-prone",
            "No validation"
        ],
        "best_for": "Prototyping, simple cases, local models"
    },
    
    "RAG (Retrieval Augmented Generation)": {
        "how_it_works": "Retrieve relevant docs, include in context",
        "pros": [
            "Simple to implement",
            "Good for knowledge retrieval",
            "No tool calling complexity"
        ],
        "cons": [
            "Limited to retrieval",
            "Can't take actions",
            "Context window limits",
            "Not for dynamic tools"
        ],
        "best_for": "Knowledge bases, Q&A, document search"
    }
}

print("COMPARISON OF APPROACHES")
print("=" * 60)
print()

for approach, details in approaches.items():
    print(f"🔧 {approach}")
    print("-" * 60)
    print(f"How it works: {details['how_it_works']}")
    print()
    print("Pros:")
    for pro in details['pros']:
        print(f"  ✓ {pro}")
    print()
    print("Cons:")
    for con in details['cons']:
        print(f"  ✗ {con}")
    print()
    print(f"Best for: {details['best_for']}")
    print()
    print()

# ============================================================================
# TODO: Choose your approach
# ============================================================================

print("=" * 60)
print("YOUR DECISION")
print("=" * 60)
print()

your_decision = """
# CHOOSE YOUR APPROACH FOR YOUR RESEARCH AGENT

For my research agent project, I will use:
Hybrid approach: Function Calling + RAG (with light Prompt Engineering in early phase)

## Reasoning

Phase 1 (Prototyping):
  Approach: Prompt Engineering + basic RAG
  Why: Fast iteration, no heavy setup, easy to test workflows and refine tool design

Phase 2 (Development):
  Approach: Function Calling + RAG
  Why: 
    - Function calling ensures reliable tool execution (search, summarize, extract)
    - RAG provides grounded, up-to-date knowledge from retrieved documents

Phase 3 (Production - if applicable):
  Approach: Function Calling + RAG + (optional MCP layer)
  Why:
    - Function calling for stable orchestration
    - RAG for scalable knowledge retrieval
    - MCP (optional) for extensibility and integration across multiple systems

## Hybrid Approach?

Yes:
- RAG handles document retrieval (papers, sources)
- Function calling handles structured operations (summarization, extraction, citations)
- Prompt engineering is used to guide reasoning and fallback behavior

## Trade-offs Accepted

- Increased system complexity compared to prompt-only solutions
- Higher implementation cost and setup time
- Need for maintaining multiple components (retriever + tools)

## Migration Path

1. Start with Prompt Engineering:
   - Simulate tool usage
   - Validate workflow and logic

2. Introduce RAG:
   - Replace manual context with real retrieval
   - Improve factual grounding

3. Add Function Calling:
   - Convert simulated tools into real APIs
   - Improve reliability and structure

4. (Optional) Introduce MCP:
   - Standardize tool interfaces
   - Scale to multi-agent or multi-system environments
"""
print(your_decision)

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Approach Selection

**Primary approach:** Function Calling + RAG (hybrid)

**Why this choice?**
Function Calling provides strict structure, reliability, and controllable execution for tools (search, summarize, extract, citations), while RAG ensures the agent is grounded in real external knowledge (especially research papers and documents). This combination balances correctness, scalability, and practical usability. Prompt engineering alone is too unstable, and MCP is more suited for larger multi-system ecosystems than this scope.

### Context Matters

**For rapid prototyping, I'd use:** Prompt Engineering  
**For production deployment, I'd use:** Function Calling + RAG  
**For academic research, I'd use:** RAG + Prompt Engineering  
**For personal projects, I'd use:** Prompt Engineering or lightweight RAG

Different contexts require different trade-offs:
- Prototyping prioritizes speed over reliability
- Production prioritizes structure and robustness
- Academic research prioritizes grounding and factual accuracy
- Personal projects prioritize simplicity and low setup cost

### MCP Specifically

**Will you use MCP in your project?** No (not initially)

**If No:**
- Why not? MCP adds infrastructure complexity that is unnecessary for a single-agent research system. It is more valuable in multi-tool, multi-agent, or enterprise environments.
- What instead? Direct Function Calling API layer with a lightweight tool registry.

### Ecosystem Consideration

**How important is reusability?**
Moderately important — tools like search, summarization, and citation generation should be reusable across workflows, but the system does not require cross-project portability at this stage.

**How important is standardization?**
Somewhat important — function schemas provide sufficient standardization without requiring full MCP adoption.

### Learning Curve

**How comfortable are you with your chosen approach?** 4/5

**What do you need to learn?**
- Advanced function calling orchestration patterns
- RAG pipeline optimization (chunking, embedding, retrieval tuning)
- Evaluation of retrieval quality and hallucination reduction
- Tool error handling in production-grade systems

**Learning resources:**
- OpenAI / LLM function calling documentation
- LangChain or LlamaIndex RAG tutorials
- Research papers on retrieval-augmented generation
- Open-source agent frameworks for architecture reference

### Future-Proofing

**Will your choice scale as your project grows?**
Yes — the hybrid approach scales well because:
- Function Calling scales across tools and workflows
- RAG scales across knowledge domains and datasets

**What might force you to change approaches?**
- Need for multi-agent orchestration (would push toward MCP)
- Need for cross-platform tool interoperability
- Enterprise-level tool sharing across systems

### Biggest Insight

**Most important factor in choosing an approach:**
Reliability vs complexity trade-off — the best architecture is not the most advanced one, but the one that minimizes failure points while meeting system needs.

**Biggest surprise:**
MCP is powerful but often unnecessary for small-to-medium agent systems; Function Calling + RAG already covers most real-world use cases effectively.

### Action Items

**Next steps to implement your chosen approach:**
1. Build a basic RAG pipeline (document ingestion + embedding + retrieval)
2. Define function calling schemas for core tools (search, summarize, extract, cite)
3. Integrate LLM orchestration layer to route between tools and RAG results
"""

print(reflection)

append_to_reflection(
    notebook="07",
    section_title="Task 3 - MCP vs Alternatives",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 3: MCP vs Alternatives

COMPARISON OF APPROACHES

🔧 MCP (Model Context Protocol)
------------------------------------------------------------
How it works: Standardized protocol, modular servers

Pros:
  ✓ Standardized - reusable across applications
  ✓ Modular - plug and play servers
  ✓ Growing ecosystem of servers
  ✓ Clean separation of concerns

Cons:
  ✗ Requires MCP infrastructure
  ✗ Still relatively new
  ✗ Need to learn MCP SDK

Best for: Production apps, reusable tools, Claude Desktop integration


🔧 Function Calling (API)
------------------------------------------------------------
How it works: Define functions in API call, LLM returns function calls

Pros:
  ✓ Built into Claude API
  ✓ No extra infrastructure
  ✓ Full control over execution
  ✓ Well documented

Cons:
  ✗ Custom code for each tool
  ✗ Not reusable across apps
  ✗ Have to handle all execution logic

Best for: API-based apps, custom tools, simple workflows


🔧 Prompt Engineering (Manual)
---------------

---
## 📚 MCP Resources & Best Practices

In [10]:
# Cell 10: MCP Resources and Best Practices
print("=" * 60)
print("MCP RESOURCES & BEST PRACTICES")
print("=" * 60)
print()

resources = {
    "Official Documentation": [
        "MCP Specification: https://modelcontextprotocol.io/specification/2025-11-25",
        "MCP SDK (TypeScript): https://github.com/modelcontextprotocol/typescript-sdk",
        "MCP SDK (Python): https://github.com/modelcontextprotocol/python-sdk",
        "Official Servers: https://github.com/modelcontextprotocol/servers"
    ],
    
    "Getting Started": [
        "Quickstart Guide: https://modelcontextprotocol.io/quickstart",
        "Building Servers: https://modelcontextprotocol.io/docs/building-servers",
        "Claude Desktop Integration: https://modelcontextprotocol.io/docs/tools/claude-desktop"
    ],
    
    "Community": [
        "Contributing: https://modelcontextprotocol.io/community/contributing",
        "Example Servers: https://github.com/modelcontextprotocol/servers/tree/main/src"
    ]
}

print("📚 LEARNING RESOURCES")
print("=" * 60)
print()

for category, links in resources.items():
    print(f"{category}:")
    for link in links:
        print(f"  • {link}")
    print()

print()
print("=" * 60)
print("BEST PRACTICES")
print("=" * 60)
print()

best_practices = {
    "Tool Design": [
        "✓ Keep tools focused - one tool, one job",
        "✓ Use clear, descriptive names",
        "✓ Document parameters thoroughly",
        "✓ Return structured data when possible",
        "✓ Include error messages in responses",
        "✗ Don't make tools too complex",
        "✗ Don't return unstructured blobs of text"
    ],
    
    "Server Development": [
        "✓ Handle errors gracefully",
        "✓ Validate inputs before processing",
        "✓ Log tool calls for debugging",
        "✓ Make servers stateless when possible",
        "✓ Version your tools",
        "✗ Don't assume perfect inputs",
        "✗ Don't store sensitive data"
    ],
    
    "Agent Design": [
        "✓ Limit maximum tool call depth",
        "✓ Track tool call costs",
        "✓ Show progress to users",
        "✓ Implement timeouts",
        "✓ Have fallback strategies",
        "✗ Don't allow infinite loops",
        "✗ Don't call expensive tools unnecessarily"
    ],
    
    "Security": [
        "✓ Validate all tool inputs",
        "✓ Use environment variables for secrets",
        "✓ Limit file system access",
        "✓ Sanitize user inputs",
        "✓ Implement rate limiting",
        "✗ Don't expose sensitive APIs",
        "✗ Don't trust tool outputs blindly"
    ]
}

for category, practices in best_practices.items():
    print(f"🎯 {category}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("GOLDEN RULES FOR MCP")
print("=" * 60)
print()
print("1. Tools should be simple, focused, and reliable")
print("2. Always validate inputs and handle errors")
print("3. Make tools reusable across projects")
print("4. Document thoroughly - your future self will thank you")
print("5. Test tools independently before integrating")
print()

MCP RESOURCES & BEST PRACTICES

📚 LEARNING RESOURCES

Official Documentation:
  • MCP Specification: https://modelcontextprotocol.io/specification/2025-11-25
  • MCP SDK (TypeScript): https://github.com/modelcontextprotocol/typescript-sdk
  • MCP SDK (Python): https://github.com/modelcontextprotocol/python-sdk
  • Official Servers: https://github.com/modelcontextprotocol/servers

Getting Started:
  • Quickstart Guide: https://modelcontextprotocol.io/quickstart
  • Building Servers: https://modelcontextprotocol.io/docs/building-servers
  • Claude Desktop Integration: https://modelcontextprotocol.io/docs/tools/claude-desktop

Community:
  • Contributing: https://modelcontextprotocol.io/community/contributing
  • Example Servers: https://github.com/modelcontextprotocol/servers/tree/main/src


BEST PRACTICES

🎯 Tool Design
------------------------------------------------------------
  ✓ Keep tools focused - one tool, one job
  ✓ Use clear, descriptive names
  ✓ Document parameters thorough

---
## ✅ Notebook 07 Complete!

### Summary

You've learned about MCP and agentic tool use! You now know:
- ✅ What MCP is and why it matters
- ✅ MCP architecture and components
- ✅ How to configure MCP servers
- ✅ Available MCP servers you can use
- ✅ How LLMs call tools
- ✅ How to design custom tools
- ✅ Tool calling strategies
- ✅ MCP vs alternative approaches
- ✅ Best practices for tool development

In [11]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Final reflection on MCP
# ============================================================================

reflection = """
### 1. How has MCP changed your view of what LLMs can do?

MCP makes LLMs feel less like standalone models and more like orchestrators inside a larger system. Instead of just generating text or calling isolated functions, they can now interact with entire ecosystems of tools, data sources, and services in a standardized way. This shifts the mental model from “LLM as a model” to “LLM as a system controller.”

### 2. Will you use MCP in your research agent?

No (for the initial version).  
The research agent can be fully built using Function Calling + RAG, which is simpler, more direct, and easier to debug. MCP would introduce extra abstraction and infrastructure complexity that is not necessary unless the system grows into a multi-agent or enterprise-scale environment.

### 3. Most exciting MCP capability?

The most exciting capability is tool interoperability at scale — the idea that any model can plug into any tool ecosystem without custom integration work. This enables reusable “tool servers” for search, memory, computation, and data access that can be shared across different agents and applications.

### 4. Biggest concern about agentic systems?

The biggest concern is uncontrolled or inefficient tool usage — agents making too many calls, retrieving low-quality data, or chaining actions incorrectly. This can lead to high cost, latency, and unreliable outputs if not carefully constrained with strong orchestration logic.

### 5. Confidence in building MCP tools? (1-5)

**Confidence:** 3/5

I understand the conceptual model, but I would need more hands-on experience with MCP server implementation, tool registration, and debugging multi-component systems. Confidence would increase with a small working prototype and exposure to real MCP ecosystems.

### 6. MCP vs Function Calling - which will you use?

Function Calling + RAG.

Reasoning:
- Function Calling provides direct control over tool execution and clear schemas
- RAG ensures strong grounding in real data
- MCP is powerful but better suited for larger distributed ecosystems, not a single research agent

### 7. Key takeaway from this notebook?

Agent design is fundamentally about orchestration, not just capability. The real challenge is not building powerful tools, but deciding when, how, and whether to use them efficiently within a controlled workflow.
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="07",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

# Show costs
print()
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()

print()
print("=" * 60)
print("✅ NOTEBOOK 07 COMPLETE!")
print("=" * 60)
print()
print("Progress: [████████████████████░] 88% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework")
print("✓ Notebook 04: Structured Outputs")
print("✓ Notebook 05: Chain of Thought")
print("✓ Notebook 06: Model Comparison")
print("✓ Notebook 07: MCP Introduction ← YOU ARE HERE")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/08_project_kickoff.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. How has MCP changed your view of what LLMs can do?

MCP makes LLMs feel less like standalone models and more like orchestrators inside a larger system. Instead of just generating text or calling isolated functions, they can now interact with entire ecosystems of tools, data sources, and services in a standardized way. This shifts the mental model from “LLM as a model” to “LLM as a system controller.”

### 2. Will you use MCP in your research agent?

No (for the initial version).  
The research agent can be fully built using Function Calling + RAG, which is simpler, more direct, and easier to debug. MCP would introduce extra abstraction and infrastructure complexity that is not necessary unless the system grows into a multi-agent or enterprise-scale environment.

### 3. Most exciting MCP capability?

The most exciting capability is tool interoperability at scale — the idea that any model can plug into any tool ecosystem without custom integration wor